# BrainGuard: architecture screen on a free Colab GPU

Trains four more ImageNet backbones (**ResNet50, DenseNet121, MobileNetV3-Large, ConvNeXt-Tiny**) on exactly the same patient-grouped 5-fold split, augmentation, head and two-phase fine-tuning as the main EfficientNetB0 model. The comparison with the main model is therefore paired, image by image.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save*, then *Runtime → Run all*.

- Total time is about **1.5–2 hours**: about 10 minutes of downloads, 5 minutes of smoke test, then training.
- Progress is saved to Google Drive (`MyDrive/brainguard_arch`). **If Colab disconnects, run all cells again**: finished folds are skipped.
- At the end, `architecture_bundle.zip` (predictions only, under 2 MB) downloads automatically. Upload it to the chat.
- No Kaggle zip is needed: the figshare data downloads automatically.

In [8]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: set Runtime -> Change runtime type -> T4 GPU"

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [9]:
import os
WORKDIR = '/content/brainguard_arch'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/brainguard_arch'
    print('Google Drive connected - progress will be saved and can resume.')
except Exception as e:
    print('Google Drive not connected (', type(e).__name__, ') - continuing without it; keep this tab open.')
os.makedirs(WORKDIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected - progress will be saved and can resume.


In [10]:
%cd /content
!rm -rf BrainGuard-MRI-Tumor-Detection
!git clone -q https://github.com/Muneera-DSA/BrainGuard-MRI-Tumor-Detection.git
%cd /content/BrainGuard-MRI-Tumor-Detection
!pip install -q -e ".[dl]"
import sys
sys.path.insert(0, '/content/BrainGuard-MRI-Tumor-Detection/src')   # make the package importable in this session without a restart

/content
/content/BrainGuard-MRI-Tumor-Detection
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for brainguard (pyproject.toml) ... done


## 1. Download and convert the figshare data (about 900 MB, 5–10 minutes)

In [11]:
from pathlib import Path
from brainguard import figshare
raw, fig = Path('/content/data/figshare_raw'), Path('/content/data/figshare')
if not (fig / 'metadata.csv').exists():
    figshare.download(raw)
    figshare.convert(raw, fig)
print(len(list((fig / 'images').glob('*'))), 'images ready')

3064 images ready


## 2. Smoke test: every backbone, 2 folds, 3 epochs (about 5 minutes). Stops here if anything is wrong.

In [12]:
SCREEN = 'resnet50_patient densenet121_patient mobilenetv3_patient convnext_tiny_patient'
!rm -rf /content/smoke
!python -m brainguard.train --experiments {SCREEN} --figshare-dir /content/data/figshare --artifacts-dir /content/smoke --quick 2>&1 | grep -E "fold|Error|Traceback|GPU"
import glob
done = glob.glob('/content/smoke/predictions/*.csv')
assert len(done) == 4, f'Smoke test failed ({len(done)} of 4 backbones finished): copy the output above and send it back'
print('Smoke test passed for all four backbones.')

W0000 00:00:1791870854.619650   18113 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1791870854.621460   18113 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
GPU found: mixed precision on
[resnet50_patient] fold 0: acc 0.867 in 75s
[resnet50_patient] fold 1: acc 0.831 in 81s
[densenet121_patient] fold 0: acc 0.872 in 98s
[densenet121_patient] fold 1: acc 0.846 in 94s
[mobilenetv3_patient] fold 0: acc 0.843 in 59s
[mobilenetv3_patient] fold 1: acc 0.835 in 59s
[convnext_tiny_patient] fold 0: acc 0.834 in 171s
[convnext_tiny_patient] fold 1: acc 0.815 in 161s
Smoke test passed for all four backbones.


## 3. Full run: 4 backbones × 5 folds (about 1–1.5 hours)

In [6]:
!python -m brainguard.train --experiments {SCREEN} --figshare-dir /content/data/figshare --artifacts-dir "{WORKDIR}/artifacts" 2>&1 | grep -E "Epoch|fold|Error|Traceback|GPU"

W0000 00:00:1791869336.334284    9691 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1791869336.336127    9691 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
GPU found: mixed precision on
[resnet50_patient] fold 0: already done, skipping
[resnet50_patient] fold 1: already done, skipping
[resnet50_patient] fold 2: already done, skipping
[resnet50_patient] fold 3: already done, skipping
[resnet50_patient] fold 4: already done, skipping
[densenet121_patient] fold 0: already done, skipping
[densenet121_patient] fold 1: already done, skipping
[densenet121_patient] fold 2: already done, skipping
[densenet121_patient] fold 3: already done, skipping
[densenet121_patient] fold 4: already done, skipping
[mobilenetv3_patient] fold 0: already don

## 4. Download `architecture_bundle.zip` and upload it to the chat

In [7]:
# Self-contained, so it also works after a Colab restart
import os, zipfile
from pathlib import Path
SCREEN = 'resnet50_patient densenet121_patient mobilenetv3_patient convnext_tiny_patient'
WORKDIR = '/content/drive/MyDrive/brainguard_arch' if os.path.isdir('/content/drive/MyDrive') else '/content/brainguard_arch'
art = Path(WORKDIR) / 'artifacts' / 'predictions'
out = Path(WORKDIR) / 'architecture_bundle.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in SCREEN.split():
        files_ = [art / f'{name}.csv', *sorted((art / name).glob('fold*.csv')), *sorted((art / name).glob('fold*_history.json'))]
        done = len(list((art / name).glob('fold*_history.json')))
        print(f'{name}: {done}/5 folds finished')
        for f in files_:
            if f.exists():
                z.write(f, f'predictions/{f.relative_to(art)}')
print(out, round(out.stat().st_size / 1e6, 2), 'MB')
from google.colab import files
files.download(str(out))

resnet50_patient: 5/5 folds finished
densenet121_patient: 5/5 folds finished
mobilenetv3_patient: 5/5 folds finished
convnext_tiny_patient: 5/5 folds finished
/content/drive/MyDrive/brainguard_arch/architecture_bundle.zip 0.5 MB
